# Testing Augmented Model on Segmented Data

This notebook brings together the final augmented model from our trials and compares it's performance on the YOLOv8 segmented leaf images

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import shutil
import zipfile
import os

shutil.copy("/content/drive/MyDrive/DLProjectData/RoCoLeDataset.zip", "/content/RoCoLeDataset.zip")
shutil.copy("/content/drive/MyDrive/DLProjectData/RoCoLeDatasetNormed.zip", "/content/RoCoLeDatasetNormed.zip")
shutil.copy("/content/drive/MyDrive/DLProjectData/CLRDataset.zip", "/content/CLRDataset.zip")
shutil.copy("/content/drive/MyDrive/DLProjectData/CLRDatasetNormed.zip", "/content/CLRDatasetNormed.zip")
shutil.copy("/content/drive/MyDrive/DLProjectData/CLRDatasetNormedAll.zip", "/content/CLRDatasetNormedAll.zip")

with zipfile.ZipFile("/content/RoCoLeDataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/RoCoLeDataset_Local")

with zipfile.ZipFile("/content/RoCoLeDatasetNormed.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/RoCoLeDatasetNormed_Local")

with zipfile.ZipFile("/content/CLRDataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/CLRDataset_Local")

with zipfile.ZipFile("/content/CLRDatasetNormed.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/CLRDatasetNormed_Local")

with zipfile.ZipFile("/content/CLRDatasetNormedAll.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/CLRDatasetNormedAll_Local")

clean_data_path = "/content/RoCoLeDataset_Local/RoCoLeDataset"
segmented_clean_data_path = "/content/RoCoLeDatasetNormed_Local/RoCoLeDatasetNormed"
messy_data_path = "/content/CLRDataset_Local/CLRDataset"
segmented_1Leaf_data_path = "/content/CLRDatasetNormed_Local/CLRDatasetNormed"
segmented_AllLeaf_data_path = "/content/CLRDatasetNormedAll_Local/CLRDatasetNormedAll"

In [3]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import Subset
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

torch.manual_seed(1234)

val_transforms = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# These are the two training datasets we want to compare.
# RoCoLeDataset = original / unnormalized leaves
# RoCoLeDatasetNormed = normalized / segmented leaves
training_configs = {
    "RoCoLeDataset": clean_data_path,
    "RoCoLeDatasetNormed": segmented_clean_data_path,
}


def make_train_val_loaders(data_path, batch_size=32, test_size=0.20, random_state=42, num_workers=2):
    """Create train/validation loaders from one ImageFolder root using a stratified split."""
    full_train_dataset = datasets.ImageFolder(data_path, transform=val_transforms)
    full_val_dataset = datasets.ImageFolder(data_path, transform=val_transforms)

    targets = full_train_dataset.targets

    train_idx, val_idx = train_test_split(
        range(len(targets)),
        test_size=test_size,
        random_state=random_state,
        stratify=targets
    )

    train_dataset = Subset(full_train_dataset, train_idx)
    test_dataset = Subset(full_val_dataset, val_idx)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    return train_dataset, test_dataset, train_loader, test_loader, full_train_dataset.classes


for dataset_name, data_path in training_configs.items():
    temp_dataset = datasets.ImageFolder(data_path, transform=val_transforms)
    print(f"{dataset_name}: {len(temp_dataset)} images, classes = {temp_dataset.classes}")

RoCoLeDataset: 1393 images, classes = ['coffee___healthy', 'coffee___rust']
RoCoLeDatasetNormed: 721 images, classes = ['coffee___healthy', 'coffee___rust']


In [4]:
eval_data = datasets.ImageFolder(root=messy_data_path, transform=val_transforms)


def make_binary_target_transform(dataset):
    def binary_target_transform(label_idx):
        class_name = dataset.classes[label_idx]
        return 0 if class_name == '0' else 1
    return binary_target_transform

eval_data.target_transform = make_binary_target_transform(eval_data)

drive_loader = DataLoader(
    eval_data,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

In [5]:
drive2_eval_data = datasets.ImageFolder(root=segmented_1Leaf_data_path, transform=val_transforms)
drive2_eval_data.target_transform = make_binary_target_transform(drive2_eval_data)

drive2_loader = DataLoader(
    drive2_eval_data,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

In [6]:
from PIL import Image
from pathlib import Path

drive3_root = Path(segmented_AllLeaf_data_path)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def collect_drive3_folders(root):
    plant_folders = []

    for class_dir in sorted(root.iterdir()):
        if not class_dir.is_dir():
            continue

        class_name = class_dir.name
        binary_label = 0 if class_name == "0" else 1

        for plant_dir in sorted(class_dir.iterdir()):
            if not plant_dir.is_dir():
                continue

            image_paths = [
                p for p in plant_dir.rglob("*")
                if p.suffix.lower() in image_extensions
            ]

            if len(image_paths) > 0:
                plant_folders.append({
                    "plant_dir": plant_dir,
                    "label": binary_label,
                    "image_paths": image_paths
                })

    return plant_folders

drive3_folders = collect_drive3_folders(drive3_root)

In [7]:
def evaluate_loader(model, dataloader, dataset_size, criterion, device):
    """Evaluate loss and accuracy on a normal image-level DataLoader."""
    model.eval()
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / dataset_size
    epoch_acc = running_corrects.double() / dataset_size

    return epoch_loss, epoch_acc.cpu().item()

In [8]:
def evaluate_drive3_folder_level(model, drive3_folders, transform, device, batch_size=32):
    model.eval()

    correct = 0
    total = 0

    all_true_labels = []
    all_pred_labels = []

    with torch.no_grad():
        for item in drive3_folders:
            true_label = item["label"]
            image_paths = item["image_paths"]

            leaf_preds = []

            for start in range(0, len(image_paths), batch_size):
                batch_paths = image_paths[start:start + batch_size]

                images = []
                for image_path in batch_paths:
                    image = Image.open(image_path).convert("RGB")
                    image = transform(image)
                    images.append(image)

                images = torch.stack(images).to(device)

                outputs = model(images)
                preds = torch.argmax(outputs, dim=1)

                leaf_preds.extend(preds.cpu().tolist())

            folder_pred = 1 if any(pred == 1 for pred in leaf_preds) else 0

            correct += int(folder_pred == true_label)
            total += 1

            all_true_labels.append(true_label)
            all_pred_labels.append(folder_pred)

    folder_acc = correct / total

    return folder_acc, all_true_labels, all_pred_labels

In [9]:
train_dataset, test_dataset, train_loader, test_loader, classes = make_train_val_loaders(
        data_path=clean_data_path,
        batch_size=128,
        test_size=0.20,
        random_state=42,
        num_workers=2
    )

In [10]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.densenet201(weights=None)
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 2)

#model weights saved in the shared data folder
model_path = '/content/drive/MyDrive/DLProjectData/densenet_finetuned.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval()

criterion = nn.CrossEntropyLoss()

print(f"Model loaded successfully from {model_path} and moved to {device}")

Model loaded successfully from /content/drive/MyDrive/DLProjectData/densenet_finetuned.pth and moved to cuda


In [11]:
val_loss, val_acc = evaluate_loader(model, test_loader, len(test_dataset), criterion, device)
drive_loss, drive_acc = evaluate_loader(model, drive_loader, len(eval_data), criterion, device)
drive2_loss, drive2_acc = evaluate_loader(model, drive2_loader, len(drive2_eval_data), criterion, device)
drive3_acc_score, drive3_true, drive3_pred = evaluate_drive3_folder_level(
    model=model,
    drive3_folders=drive3_folders,
    transform=val_transforms,
    device=device
)

In [12]:
import pandas as pd

results = {
    "Evaluation Set": [
        "RoCoLe (Validation)", 
        "Messy Field Data (Raw)", 
        "Messy Field Data (Segmented)", 
        "Folder-Level (All Leaves)"
    ],
    "Loss": [
        f"{val_loss:.4f}", 
        f"{drive_loss:.4f}", 
        f"{drive2_loss:.4f}", 
        "N/A"
    ],
    "Accuracy (%)": [
        f"{val_acc * 100:.2f}%", 
        f"{drive_acc * 100:.2f}%", 
        f"{drive2_acc * 100:.2f}%", 
        f"{drive3_acc_score * 100:.2f}%"
    ]
}

df_results = pd.DataFrame(results)
display(df_results)

,Evaluation Set,Loss,Accuracy (%)
0,RoCoLe (Validation),0.1443,95.34%
1,Messy Field Data (Raw),0.4070,82.92%
2,Messy Field Data (Segmented),0.6371,66.23%
3,Folder-Level (All Leaves),N/A,40.44%
